In [1]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('dataset/email_spam_100k.csv')

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['body'] = df['body'].apply(clean_text)

In [2]:
X = df['body']
y = df['label']

X_train1, X_test1, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),     
    max_df=0.9,
    min_df=2,             
    sublinear_tf=True,      
    max_features=40000     
)

In [3]:
X_train = vectorizer.fit_transform(X_train1)
X_test = vectorizer.transform(X_test1)

model = LogisticRegression(
    max_iter=1000,
    C=3,  # slightly tuned
    class_weight='balanced'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print(" Accuracy:", round(accuracy, 4))
print(" Precision (Spam):", round(precision, 4))
print("\n Report:\n", classification_report(y_test, y_pred))


 Accuracy: 0.975
 Precision (Spam): 0.9674

 Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.98     10443
           1       0.97      0.98      0.97      9557

    accuracy                           0.98     20000
   macro avg       0.97      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000



In [4]:
pickle.dump(model, open("lr.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))